# 실험 01: JsonOutputParser (전통적인 JSON 파싱)

## 1. 실험 제목과 목표
- **제목**: 프롬프트를 통한 JSON 구조화 및 파싱
- **목표**: 자연어를 나중에 파싱하는 구식 방법(Output Parser) 중 가장 널리 쓰이는 `JsonOutputParser`의 원리를 이해하고, 이 방식이 가지는 취약점(에러 케이스)을 직접 관찰합니다.

## 2. 실험 개요
1. **비교 실험 1**: 파서 없이 프롬프트로만 "JSON으로 줘"라고 했을 때 vs `JsonOutputParser`를 썼을 때의 차이
2. **실패/주의 케이스**: 모델이 JSON 형식(따옴표, 괄호 등)을 틀리거나 앞뒤에 쓸데없는 인사말을 붙일 때 터지는 `OutputParserException` 관찰

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

## 4. 환경 변수 로드 및 모델 준비

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## 5. 비교 실험 1: 프롬프트 의존 vs JsonOutputParser
LLM에게 데이터베이스에 넣을 JSON을 요구해봅시다.

In [3]:
question = "이순신 장군에 대해 이름(name), 시대(era), 업적 1개(achievement)를 JSON으로 줘."

print("=== [1. 파서 없이 프롬프트로만 요구 (문자열 반환)] ===")
raw_chain = PromptTemplate.from_template("{query}") | llm
raw_res = raw_chain.invoke({"query": question})
print("타입:", type(raw_res.content))
print("내용:\n", raw_res.content)
# 문자열이므로 Python 딕셔너리처럼 res['name'] 형식으로 꺼내 쓸 수 없습니다. 수동으로 json.loads()를 해야 합니다.

print("\n=== [2. JsonOutputParser 도입 (딕셔너리 반환)] ===")
json_parser = JsonOutputParser()

# 파서가 스스로 "JSON으로 대답해"라는 프롬프트를 만들어줍니다.
format_instructions = json_parser.get_format_instructions()
print("-> 파서 자동 생성 지시문:", format_instructions)

json_prompt = PromptTemplate(
    template="{query}\n\n{format_instructions}",
    input_variables=["query"],
    partial_variables={"format_instructions": format_instructions}
)

json_chain = json_prompt | llm | json_parser
parsed_res = json_chain.invoke({"query": question})
print("\n타입:", type(parsed_res))
print("내용:", parsed_res)
print("이름 꺼내기:", parsed_res.get("name", "이름 없음"))

print("\n-> 결과: JsonOutputParser가 모델 응답을 파이썬 dict로 변환해주어 애플리케이션에서 즉시 사용할 수 있게 됩니다.")

=== [1. 파서 없이 프롬프트로만 요구 (문자열 반환)] ===
타입: <class 'str'>
내용:
 ```json
{
  "name": "이순신",
  "era": "조선시대",
  "achievement": "임진왜란 당시 일본 군대를 상대로 한 여러 차례의 해전에서의 승리"
}
```

=== [2. JsonOutputParser 도입 (딕셔너리 반환)] ===
-> 파서 자동 생성 지시문: Return a JSON object.

타입: <class 'dict'>
내용: {'name': '이순신', 'era': '조선 시대', 'achievement': '명량해전에서 대승을 거두어 일본 함대의 침공을 저지함'}
이름 꺼내기: 이순신

-> 결과: JsonOutputParser가 모델 응답을 파이썬 dict로 변환해주어 애플리케이션에서 즉시 사용할 수 있게 됩니다.


## 6. 실패/주의 케이스: 인사말이나 마크다운이 섞였을 때
모델이 친절하게 "네, 알겠습니다!"라는 말을 덧붙이거나 JSON 포맷을 실수하면 파서는 터져버립니다.

In [4]:
print("[모델이 친절함을 발휘하는 상황 시뮬레이션]")
bad_llm_response = """네, 알겠습니다! 요청하신 JSON입니다.
{
  "name": "이순신",
  "era": "조선"
}
도움이 되셨길 바랍니다!"""

try:
    # 강제로 파서에 나쁜 응답을 먹여봅니다.
    json_parser.parse(bad_llm_response)
except Exception as e:
    print("🔥 에러 발생:", type(e).__name__)
    print("에러 메시지:", str(e))
    print("\n-> 주의: 최신 LangChain 파서는 마크다운(```json) 정도는 벗겨내 주지만, 위처럼 사람의 말이 앞뒤에 심하게 섞여 있으면 여지없이 OutputParserException이 발생합니다.")

[모델이 친절함을 발휘하는 상황 시뮬레이션]
🔥 에러 발생: OutputParserException
에러 메시지: Invalid json output: 네, 알겠습니다! 요청하신 JSON입니다.
{
  "name": "이순신",
  "era": "조선"
}
도움이 되셨길 바랍니다!
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

-> 주의: 최신 LangChain 파서는 마크다운(```json) 정도는 벗겨내 주지만, 위처럼 사람의 말이 앞뒤에 심하게 섞여 있으면 여지없이 OutputParserException이 발생합니다.


## 7. 결과 해석

1. **포맷팅 자동화**: `JsonOutputParser`는 `get_format_instructions()`를 통해 모델에게 'JSON으로 대답하라'는 프롬프트를 자동으로 달아주는 역할을 겸합니다.
2. **파싱 취약성**: 자연어 생성 후 -> JSON 변환을 시도하는 2단계 접근법이므로, 모델의 컨디션(친절함 등)에 따라 파싱이 붕괴될 위험을 항상 안고 있습니다.
3. **타입의 부재**: 추출된 결과는 dict이지만, `name`이 문자열인지, `era`가 숫자인지 강제(Validation)할 수 없습니다.

## 8. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* `JsonOutputParser`를 쓰면 Python Dictionary로 깔끔하게 떨어져서 꺼내 쓰기 좋음.
* 하지만 필드명(`name`, `era`)을 모델 맘대로 바꿔 부를 수도 있고, 타입 검사도 안 됨.
* 무엇보다 모델이 앞뒤에 이상한 말을 덧붙이는 순간 시스템이 뻗어버리는 치명적 약점 관찰.

### 다음 개선 방향
* 자유로운 dict 형태 대신, 파이썬의 엄격한 클래스(타입 힌트 적용)를 정의해서 모델이 그 양식에만 맞춰서 답을 내도록 강제하는 `Pydantic` 도입 필요.